# Baseline: D3

D3 (Bao et al., EMNLP 2024) usa el modelo de BIGRec con otra decodificación: `length_penalty = 0` en el beam search, sin la normalización por largo. No se reentrena y no usa el TFA (mezcla de logits de un modelo tradicional), coherente con el Apéndice B del paper. El código de generación es el de `../bigrec/bigrec_baseline.py` con `variante="d3"`.

Escribe las predicciones en `checkpoints/bigrec/<categoría>/predicciones/d3/`, junto a las de BIGRec, y la evaluación se puede retomar si se corta. Las decisiones, los costos y los resultados están en [README.md](README.md).


## Entorno y configuración

In [1]:
import os
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if not IN_COLAB:
    os.environ['HF_HUB_OFFLINE'] = '1'   # el modelo se carga desde checkpoints/: no se descarga nada
    # Solo la GPU dedicada (índice 0), antes de importar torch; no combinar con HIP_VISIBLE_DEVICES
    os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')

import sys
import json
import pandas as pd
import torch
import transformers

E-001h rocSHMEM Could not open libnuma. Returning	NUMAWrapper@src/gda/numa_wrapper.cpp:48


In [2]:
DATASETS = ['toys_data']   # se evalúan solo estos (nombres de `names`); necesitan el modelo de BIGRec entrenado
names = {'CDs_and_Vinyl': 'cd_data',
         'Musical_Instruments': 'music_data',
         'Toys_and_Games': 'toys_data',
         'Video_Games': 'games_data'}
categorias = {nombre: categoria for categoria, nombre in names.items()}

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
base = 'data_amazon_2018' if IN_COLAB else os.path.join('..', '..', 'repo_latentr3', 'data')
codigo_d3 = os.path.abspath('DecodingMatters' if IN_COLAB else os.path.join('..', '..', 'DecodingMatters'))
latentr3 = os.path.abspath('repo_latentr3' if IN_COLAB else os.path.join('..', '..', 'repo_latentr3'))
# Modelos y predicciones de BIGRec y D3 comparten carpeta
SALIDA = ('/content/drive/MyDrive/IIC3633/checkpoints/bigrec' if IN_COLAB
          else os.path.abspath(os.path.join('..', '..', 'checkpoints', 'bigrec')))
calc_py = os.path.join(latentr3, 'src', 'utils', 'calc.py')
# bigrec_baseline.py (notebooks/bigrec/) y baseline_comun.py (notebooks/); en Colab, en el directorio de trabajo
modulos = [os.path.abspath('.')] if IN_COLAB else [os.path.abspath(os.path.join('..', 'bigrec')), os.path.abspath('..')]
for carpeta in modulos:
    if carpeta not in sys.path:
        sys.path.insert(0, carpeta)

In [3]:
for archivo in ['code/dataset.py', 'code/LogitProcesser.py']:
    if not os.path.isfile(os.path.join(codigo_d3, archivo)):
        raise FileNotFoundError(f'No se encontró {archivo} en {codigo_d3}. Descargar '
                                'https://github.com/SAI990323/DecodingMatters en esa carpeta y hacer checkout del commit 585c842.')
if not os.path.isfile(calc_py):
    raise FileNotFoundError(f'No se encontró {calc_py}. En Colab hay que subir repo_latentr3/src/.')
for modulo in ['bigrec_baseline.py', 'baseline_comun.py']:
    if not any(os.path.isfile(os.path.join(carpeta, modulo)) for carpeta in modulos):
        raise FileNotFoundError(f'No se encontró {modulo} en {modulos}.')

import bigrec_baseline as bb
bb.cargar_d3(codigo_d3)   # verifica el sha256 antes de importar

print('torch', torch.__version__, '| transformers', transformers.__version__)
if transformers.__version__ != '4.51.3':
    print('AVISO: los autores usan transformers 4.51.3 y advierten que otra versión cambia los resultados (issue #2).')
assert torch.cuda.is_available(), 'La evaluación necesita una GPU'
print(torch.cuda.get_device_name(0), '| bf16:', torch.cuda.is_bf16_supported())
print('length_penalty BIGRec:', bb.LENGTH_PENALTY['bigrec'], '| D3:', bb.LENGTH_PENALTY['d3'])

torch 2.14.1+rocm7.14 | transformers 4.51.3
AMD Radeon RX 9070 XT | bf16: True
length_penalty BIGRec: 1.0 | D3: 0.0


## Modelo de BIGRec

D3 usa el modelo de BIGRec de cada dataset, sin reentrenar. Se comprueba que el entrenamiento haya terminado (`entrenamiento.json` con `terminado: true`) y que exista `modelo/`, y se muestra la época elegida.

In [4]:
modelos = {}
for nombre in DATASETS:
    salida = os.path.join(SALIDA, categorias[nombre])
    if not bb.entrenado(salida) or not os.path.isfile(os.path.join(salida, 'modelo', 'config.json')):
        raise FileNotFoundError(f'Falta el modelo de BIGRec de {nombre} en {salida}. Entrenarlo con '
                                '../bigrec/baseline_bigrec.ipynb.')
    with open(os.path.join(salida, 'entrenamiento.json')) as f:
        r = json.load(f)
    modelos[nombre] = {'eval_loss por época': [round(e['eval_loss'], 4) for e in r['evaluaciones']],
                       'mejor eval_loss': r['mejor_eval_loss'],
                       'checkpoint elegido': os.path.basename(r['mejor_checkpoint'])}
pd.DataFrame(modelos).T

,eval_loss por época,mejor eval_loss,checkpoint elegido
toys_data,"[0.731, 0.5748, 0.6258]",0.57483,checkpoint-844


## Evaluación

Igual que BIGRec: para cada fila de `test` se generan 10 títulos con beam search de 10 beams, restringido por el trie a los títulos del catálogo. La única diferencia es el orden final de los beams:
- **BIGRec** (`length_penalty = 1`): log-probabilidad del título dividida por su largo en tokens.
- **D3** (`length_penalty = 0`): log-probabilidad sin dividir.

Según D3, dividir por el largo favorece los títulos largos: muchos de sus tokens finales ("ghost tokens") tienen probabilidad cercana a 1 y suben el promedio sin aportar información.

Se guarda por partes de 256 filas en `predicciones/d3/`, así que se puede retomar volviendo a correr la celda. Al final se escribe `final_result.json` con el formato que lee `calc.py`. En Toys, la evaluación de BIGRec tardó 1724,18 s.

In [5]:
for nombre in DATASETS:
    categoria = categorias[nombre]
    bb.evaluar_categoria(categoria, base, os.path.join(SALIDA, categoria), 'd3')

100%|██████████| 6738/6738 [00:01<00:00, 3372.88it/s]
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
`generation_config` default values have been modified to match model-specific defaults: {'bos_token_id': 151643}. If this is not desired, please set these values explicitly.


Toys_and_Games (d3): parte 1/27, 0.31 s por fila
Toys_and_Games (d3): parte 2/27, 0.25 s por fila
Toys_and_Games (d3): parte 3/27, 0.24 s por fila
Toys_and_Games (d3): parte 4/27, 0.32 s por fila


[W1106 07:11:02.277664484 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 1247805440 bytes (free: 490733568, total: 17095983104).


Toys_and_Games (d3): parte 5/27, 0.24 s por fila
Toys_and_Games (d3): parte 6/27, 0.26 s por fila
Toys_and_Games (d3): parte 7/27, 0.26 s por fila
Toys_and_Games (d3): parte 8/27, 0.26 s por fila
Toys_and_Games (d3): parte 9/27, 0.25 s por fila
Toys_and_Games (d3): parte 10/27, 0.25 s por fila


[W1106 07:17:27.970892738 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 1264582656 bytes (free: 631242752, total: 17095983104).


Toys_and_Games (d3): parte 11/27, 0.27 s por fila
Toys_and_Games (d3): parte 12/27, 0.26 s por fila
Toys_and_Games (d3): parte 13/27, 0.26 s por fila
Toys_and_Games (d3): parte 14/27, 0.24 s por fila
Toys_and_Games (d3): parte 15/27, 0.26 s por fila
Toys_and_Games (d3): parte 16/27, 0.25 s por fila
Toys_and_Games (d3): parte 17/27, 0.26 s por fila
Toys_and_Games (d3): parte 18/27, 0.25 s por fila
Toys_and_Games (d3): parte 19/27, 0.27 s por fila
Toys_and_Games (d3): parte 20/27, 0.26 s por fila
Toys_and_Games (d3): parte 21/27, 0.25 s por fila
Toys_and_Games (d3): parte 22/27, 0.26 s por fila
Toys_and_Games (d3): parte 23/27, 0.28 s por fila
Toys_and_Games (d3): parte 24/27, 0.26 s por fila
Toys_and_Games (d3): parte 25/27, 0.26 s por fila


[W1106 07:34:21.035337130 HIPCachingAllocator.cpp:3935] memory allocation failed with OOM on device 0 while trying to allocate 1308622848 bytes (free: 392167424, total: 17095983104).


Toys_and_Games (d3): parte 26/27, 0.26 s por fila
Toys_and_Games (d3): parte 27/27, 0.26 s por fila


## Métricas

HR@K y NDCG@K con el criterio de `calc.py`: hay acierto si el título generado, normalizado, es igual al título del objetivo. Para cada predicción, `bb.verificar_con_calc` corre `gao` de `calc.py` sobre el mismo `final_result.json` y falla si da algo distinto. Los títulos fuera del catálogo deben ser 0.

Se calculan BIGRec y D3 para todos los datasets que tengan predicciones, no solo los de `DATASETS`.

In [6]:
VARIANTES = {'bigrec': 'BIGRec', 'd3': 'D3'}
resultados, rutas = {}, {}
for nombre, categoria in categorias.items():
    info = bb.buscar_archivo(base, 'info', categoria)
    for variante, etiqueta in VARIANTES.items():
        ruta = os.path.join(SALIDA, categoria, 'predicciones', variante, 'final_result.json')
        if not os.path.isfile(ruta):
            continue
        bb.verificar_con_calc(ruta, info, calc_py)
        resultados[(nombre, etiqueta)] = bb.metricas(ruta, info)
        rutas[(nombre, variante)] = ruta
print('Evaluados:', list(resultados))
assert all(r['fuera del catálogo'] == 0 for r in resultados.values())
pd.DataFrame(resultados).T.round(4)

Evaluados: [('toys_data', 'BIGRec'), ('toys_data', 'D3')]


HR@1    HR@3    HR@5   HR@10   HR@20  NDCG@1  NDCG@3  \
toys_data BIGRec  0.0352  0.0591  0.0770  0.1060  0.1060  0.0352  0.0490   
          D3      0.0411  0.0693  0.0873  0.1153  0.1153  0.0411  0.0576   

                  NDCG@5  NDCG@10  NDCG@20  fuera del catálogo  repetidos  \
toys_data BIGRec  0.0563   0.0657   0.0657                 0.0        0.0   
          D3      0.0649   0.0740   0.0740                 0.0        0.0   

                   filas  
toys_data BIGRec  6738.0  
          D3      6738.0

### Efecto del cambio de decodificación

Como el modelo y los beams son los mismos, D3 solo reordena o reemplaza títulos respecto de BIGRec. Se cuenta cuántas filas cambian su top-10 y su top-1, y el largo medio (en caracteres) del título en primer lugar. Si D3 corrige el sesgo hacia títulos largos, su top-1 debería ser más corto.

In [7]:
efecto = {}
for nombre in categorias:
    if (nombre, 'bigrec') not in rutas or (nombre, 'd3') not in rutas:
        continue
    pred = {}
    for variante in VARIANTES:
        with open(rutas[(nombre, variante)]) as f:
            pred[variante] = [fila['predict'] for fila in json.load(f)]
    b, d = pred['bigrec'], pred['d3']
    efecto[nombre] = {'filas': len(b),
                      'top-10 distinto (%)': 100 * sum(x != y for x, y in zip(b, d)) / len(b),
                      'conjunto top-10 distinto (%)': 100 * sum(set(x) != set(y) for x, y in zip(b, d)) / len(b),
                      'top-1 distinto (%)': 100 * sum(x[0] != y[0] for x, y in zip(b, d)) / len(b),
                      'largo top-1 BIGRec': sum(len(x[0].strip()) for x in b) / len(b),
                      'largo top-1 D3': sum(len(y[0].strip()) for y in d) / len(d)}
pd.DataFrame(efecto).T.round(2)

,filas,top-10 distinto (%),conjunto top-10 distinto (%),top-1 distinto (%),largo top-1 BIGRec,largo top-1 D3
toys_data,6738.0,99.96,84.86,58.44,74.18,52.71


## Comparación con el paper

Tabla 1 del paper, filas BIGRec y D3 (una corrida determinista, Apéndice E). En el paper, D3 mejora a BIGRec en los cuatro datasets. Aquí, BIGRec y D3 usan el mismo modelo, así que la diferencia entre "Nuestro BIGRec" y "Nuestro D3" es solo la decodificación.

El paper no especifica si los autores reentrenaron un modelo aparte para D3, si usaron exactamente `length_penalty = 0.0`, ni los `max_new_tokens` y el batch de esa corrida.


In [8]:
METRICAS = ['HR@5', 'HR@10', 'NDCG@5', 'NDCG@10']
PAPER = {'BIGRec': {'toys_data': [0.0701, 0.0931, 0.0508, 0.0582], 'cd_data': [0.0757, 0.0929, 0.0616, 0.0672],
                    'games_data': [0.0461, 0.0709, 0.0334, 0.0414], 'music_data': [0.0938, 0.1158, 0.0807, 0.0879]},
         'D3': {'toys_data': [0.0830, 0.1026, 0.0610, 0.0674], 'cd_data': [0.1122, 0.1272, 0.0906, 0.0955],
                'games_data': [0.0608, 0.0860, 0.0423, 0.0505], 'music_data': [0.0984, 0.1167, 0.0848, 0.0907]}}
filas = []
for nombre in categorias:
    if not any((nombre, etiqueta) in resultados for etiqueta in VARIANTES.values()):
        continue
    for etiqueta in VARIANTES.values():
        filas.append({'dataset': nombre, 'fuente': f'Tabla 1: {etiqueta}', **dict(zip(METRICAS, PAPER[etiqueta][nombre]))})
    for etiqueta in VARIANTES.values():
        if (nombre, etiqueta) in resultados:
            filas.append({'dataset': nombre, 'fuente': f'Nuestro {etiqueta}',
                          **{m: resultados[(nombre, etiqueta)][m] for m in METRICAS}})
comparacion = pd.DataFrame(filas).set_index(['dataset', 'fuente'])
comparacion.round(4)

HR@5   HR@10  NDCG@5  NDCG@10
dataset   fuente                                          
toys_data Tabla 1: BIGRec  0.0701  0.0931  0.0508   0.0582
          Tabla 1: D3      0.0830  0.1026  0.0610   0.0674
          Nuestro BIGRec   0.0770  0.1060  0.0563   0.0657
          Nuestro D3       0.0873  0.1153  0.0649   0.0740

In [9]:
# Mejora relativa de D3 sobre BIGRec (%), en el paper y en nuestra corrida
mejora = {}
for nombre in categorias:
    if (nombre, 'BIGRec') in resultados and (nombre, 'D3') in resultados:
        paper_b, paper_d = PAPER['BIGRec'][nombre], PAPER['D3'][nombre]
        mejora[(nombre, 'Tabla 1')] = {m: 100 * (d / b - 1) for m, b, d in zip(METRICAS, paper_b, paper_d)}
        mejora[(nombre, 'Nuestra')] = {m: 100 * (resultados[(nombre, 'D3')][m] / resultados[(nombre, 'BIGRec')][m] - 1)
                                       for m in METRICAS}
pd.DataFrame(mejora).T.round(1)

HR@5  HR@10  NDCG@5  NDCG@10
toys_data Tabla 1  18.4   10.2    20.1     15.8
          Nuestra  13.3    8.8    15.2     12.7